First installing the pdfplumber

In [23]:
%pip install -q pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 113.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 103.0 MB/s eta 0:00:00


now making a function which uploads the PDF from your machine into the Colab runtime.

In [24]:
from google.colab import files

uploaded = files.upload()
pdf_name = list(uploaded.keys())[0]
print("Uploaded file:", pdf_name)

Saving 10 RAG Architectures.pdf to 10 RAG Architectures.pdf
Uploaded file: 10 RAG Architectures.pdf


now checkng that pdfplumber can read this file or not.

In [25]:
import pdfplumber
with pdfplumber.open(pdf_name) as pdf:
  print("Number of pages =", len(pdf.pages))

Number of pages = 8


Extracting the text from each page, splitter.

In [26]:
#all_text → stores full combined text of the PDF
all_text = ""
page_data = []
#page_data → stores structured info for each page
with pdfplumber.open(pdf_name) as pdf:
    for i, page in enumerate(pdf.pages):
        text = page.extract_text()
        cleaned_text = text.strip() if text else ""
        # page.extract_text() → tries to pull text from that page
        # cleaned_text → removes useless surrounding whitespace
        page_info = {
            "page_number": i + 1,
            "char_count": len(cleaned_text),
            "word_count": len(cleaned_text.split()) if cleaned_text else 0,
            "likely_scanned": len(cleaned_text) < 50,
            "preview": cleaned_text[:500] if cleaned_text else "[No extractable text found]"
        }

        page_data.append(page_info)

        if cleaned_text:
            all_text += f"\n\n--- Page {i+1} ---\n{cleaned_text}"

print("Extraction complete.")
print("Pages processed:", len(page_data))
#likely_scanned → marks suspicious pages with almost no extracted text

Extraction complete.
Pages processed: 8


In [27]:
for page in page_data:
    print("=" * 70)
    print(f"Page {page['page_number']}")
    print("Character count:", page["char_count"])
    print("Word count:", page["word_count"])
    print("Likely scanned:", page["likely_scanned"])
    print("Preview:")
    print(page["preview"])
    print()

Page 1
Character count: 803
Word count: 132
Likely scanned: False
Preview:
10 RAG Architectures
1. Standard RAG
The baseline Retrieval-Augmented Generation setup.
A user query is embedded, relevant documents are retrieved from a vector
database, and the LLM generates an answer grounded in that context.
Example
User: “What are the side effects of ibuprofen?”
System:
● Retrieves medical documents
● Feeds them to the LLM
● Generates a grounded answer
Usage
● FAQ bots
● Knowledge base assistants
● Documentation search
● Customer support
Best when:
● Knowledge is static or 

Page 2
Character count: 792
Word count: 123
Likely scanned: False
Preview:
User: “Compare Tesla and BYD revenue growth over the last 3 years.”
Agent:
1. Decides to fetch financial data
2. Calls APIs / retrieves reports
3. May refine query multiple times
4. Aggregates + reasons
5. Produces final comparison
Usage
● Research assistants
● Financial analysis
● Complex Q&A requiring multiple sources
● Autonomous workflows
Bes

In [28]:
print(all_text[:50000] if all_text else "[No text extracted]")
#seeing that is the all of the document text visible.



--- Page 1 ---
10 RAG Architectures
1. Standard RAG
The baseline Retrieval-Augmented Generation setup.
A user query is embedded, relevant documents are retrieved from a vector
database, and the LLM generates an answer grounded in that context.
Example
User: “What are the side effects of ibuprofen?”
System:
● Retrieves medical documents
● Feeds them to the LLM
● Generates a grounded answer
Usage
● FAQ bots
● Knowledge base assistants
● Documentation search
● Customer support
Best when:
● Knowledge is static or slow-changing
● No multi-step reasoning required
2. Agentic RAG
RAG + autonomous decision-making. The system behaves like an agent that
can decide:
● what to retrieve
● whether to retrieve
● which tools to call
● how many steps to take
It’s iterative and dynamic rather than being a single pass.
Example

--- Page 2 ---
User: “Compare Tesla and BYD revenue growth over the last 3 years.”
Agent:
1. Decides to fetch financial data
2. Calls APIs / retrieves reports
3. May refine query

Chunking

In [29]:
%pip install -q langchain-text-splitters
#for chunking we need to add the splitters

In [30]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
#splitters will split the document into multiple chunks.

In [31]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]
)

In [32]:
chunks = splitter.split_text(all_text)

print("Total chunks:", len(chunks))
print("\nFirst chunk:\n")
print(chunks[0])

Total chunks: 16

First chunk:

--- Page 1 ---
10 RAG Architectures
1. Standard RAG
The baseline Retrieval-Augmented Generation setup.
A user query is embedded, relevant documents are retrieved from a vector
database, and the LLM generates an answer grounded in that context.
Example
User: “What are the side effects of ibuprofen?”
System:
● Retrieves medical documents
● Feeds them to the LLM
● Generates a grounded answer
Usage
● FAQ bots
● Knowledge base assistants
● Documentation search
● Customer support
Best when:


In [33]:
for i, chunk in enumerate(chunks[:5]):
    print("=" * 80)
    print(f"Chunk {i+1}")
    print("Length:", len(chunk))
    print(chunk[:700])
    print()

Chunk 1
Length: 489
--- Page 1 ---
10 RAG Architectures
1. Standard RAG
The baseline Retrieval-Augmented Generation setup.
A user query is embedded, relevant documents are retrieved from a vector
database, and the LLM generates an answer grounded in that context.
Example
User: “What are the side effects of ibuprofen?”
System:
● Retrieves medical documents
● Feeds them to the LLM
● Generates a grounded answer
Usage
● FAQ bots
● Knowledge base assistants
● Documentation search
● Customer support
Best when:

Chunk 2
Length: 426
Usage
● FAQ bots
● Knowledge base assistants
● Documentation search
● Customer support
Best when:
● Knowledge is static or slow-changing
● No multi-step reasoning required
2. Agentic RAG
RAG + autonomous decision-making. The system behaves like an agent that
can decide:
● what to retrieve
● whether to retrieve
● which tools to call
● how many steps to take
It’s iterative and dynamic rather than being a single pass.
Example

Chunk 3
Length: 479
--- Page 2 ---
User: 

In [34]:
for i in range(min(6, len(chunks) - 1)):
    print("=" * 80)
    print(f"END OF CHUNK {i+1}")
    print(chunks[i][-200:])
    print()
    print(f"START OF CHUNK {i+2}")
    print(chunks[i+1][:200])
    print()

END OF CHUNK 1
buprofen?”
System:
● Retrieves medical documents
● Feeds them to the LLM
● Generates a grounded answer
Usage
● FAQ bots
● Knowledge base assistants
● Documentation search
● Customer support
Best when:

START OF CHUNK 2
Usage
● FAQ bots
● Knowledge base assistants
● Documentation search
● Customer support
Best when:
● Knowledge is static or slow-changing
● No multi-step reasoning required
2. Agentic RAG
RAG + autonom

END OF CHUNK 2
ystem behaves like an agent that
can decide:
● what to retrieve
● whether to retrieve
● which tools to call
● how many steps to take
It’s iterative and dynamic rather than being a single pass.
Example

START OF CHUNK 3
--- Page 2 ---
User: “Compare Tesla and BYD revenue growth over the last 3 years.”
Agent:
1. Decides to fetch financial data
2. Calls APIs / retrieves reports
3. May refine query multiple times
4. Agg

END OF CHUNK 3
nancial analysis
● Complex Q&A requiring multiple sources
● Autonomous workflows
Best when:
○ Multi-step reasonin

embeddings and semantic search.

In [35]:
%pip install -q sentence-transformers scikit-learn

In [36]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [37]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [38]:
chunk_embeddings = model.encode(chunks, convert_to_numpy=True)

print("Embedding shape:", chunk_embeddings.shape)

Embedding shape: (16, 384)


In [39]:
query = "What is agentic RAG?"
query_embedding = model.encode([query], convert_to_numpy=True)

scores = cosine_similarity(query_embedding, chunk_embeddings)[0]
top_indices = np.argsort(scores)[::-1][:3]

for rank, idx in enumerate(top_indices, start=1):
    print("=" * 80)
    print(f"Rank {rank} | Chunk index: {idx} | Score: {scores[idx]:.4f}")
    print(chunks[idx][:700])
    print()

Rank 1 | Chunk index: 1 | Score: 0.3140
Usage
● FAQ bots
● Knowledge base assistants
● Documentation search
● Customer support
Best when:
● Knowledge is static or slow-changing
● No multi-step reasoning required
2. Agentic RAG
RAG + autonomous decision-making. The system behaves like an agent that
can decide:
● what to retrieve
● whether to retrieve
● which tools to call
● how many steps to take
It’s iterative and dynamic rather than being a single pass.
Example

Rank 2 | Chunk index: 0 | Score: 0.3031
--- Page 1 ---
10 RAG Architectures
1. Standard RAG
The baseline Retrieval-Augmented Generation setup.
A user query is embedded, relevant documents are retrieved from a vector
database, and the LLM generates an answer grounded in that context.
Example
User: “What are the side effects of ibuprofen?”
System:
● Retrieves medical documents
● Feeds them to the LLM
● Generates a grounded answer
Usage
● FAQ bots
● Knowledge base assistants
● Documentation search
● Customer support
Best when:

R

For ranking we will be using the BM25.

In [40]:
%pip install -q rank-bm25

In [41]:
from rank_bm25 import BM25Okapi

In [42]:
tokenized_chunks = [chunk.lower().split() for chunk in chunks]
bm25 = BM25Okapi(tokenized_chunks)
#This is a simple tokenizer:
#lowercase everything
#split by spaces

In [43]:
#Query with BM25
query = "What is agentic RAG?"
tokenized_query = query.lower().split()

bm25_scores = bm25.get_scores(tokenized_query)
top_bm25_indices = np.argsort(bm25_scores)[::-1][:3]

for rank, idx in enumerate(top_bm25_indices, start=1):
    print("=" * 80)
    print(f"Rank {rank} | Chunk index: {idx} | Score: {bm25_scores[idx]:.4f}")
    print(chunks[idx][:700])
    print()

Rank 1 | Chunk index: 1 | Score: 4.8720
Usage
● FAQ bots
● Knowledge base assistants
● Documentation search
● Customer support
Best when:
● Knowledge is static or slow-changing
● No multi-step reasoning required
2. Agentic RAG
RAG + autonomous decision-making. The system behaves like an agent that
can decide:
● what to retrieve
● whether to retrieve
● which tools to call
● how many steps to take
It’s iterative and dynamic rather than being a single pass.
Example

Rank 2 | Chunk index: 12 | Score: 0.7193
Example
User: “Who is the CEO of the company that owns Instagram?”
System:
● Uses knowledge graph to resolve relationships
● Uses LLM for explanation
● Produces accurate, structured answer
Usage
● Enterprise knowledge systems
● Compliance / regulated domains
● Complex relational queries
Best when:
○ Structured + unstructured data both matter
○ Logical reasoning is required
9. Cost-Constrained RAG

Rank 3 | Chunk index: 11 | Score: 0.6912
--- Page 6 ---
● Long-document QA
● Summarization

Now Hybrid Approach

In [44]:
#getting the top results from both.
dense_top_k = 5
bm25_top_k = 5

dense_indices = np.argsort(scores)[::-1][:dense_top_k]
bm25_indices = np.argsort(bm25_scores)[::-1][:bm25_top_k]

print("Dense top indices:", dense_indices)
print("BM25 top indices:", bm25_indices)

Dense top indices: [ 1  0 14 11  3]
BM25 top indices: [ 1 12 11 15  5]


In [45]:
#making the rank maps
dense_rank_map = {doc_id: rank + 1 for rank, doc_id in enumerate(dense_indices)}
bm25_rank_map = {doc_id: rank + 1 for rank, doc_id in enumerate(bm25_indices)}

print("Dense rank map:", dense_rank_map)
print("BM25 rank map:", bm25_rank_map)

Dense rank map: {np.int64(1): 1, np.int64(0): 2, np.int64(14): 3, np.int64(11): 4, np.int64(3): 5}
BM25 rank map: {np.int64(1): 1, np.int64(12): 2, np.int64(11): 3, np.int64(15): 4, np.int64(5): 5}


In [46]:
#computing the RRF (Reciprocal Rank Fusion)
k = 60
all_candidates = set(dense_indices).union(set(bm25_indices))

rrf_scores = {}

for doc_id in all_candidates:
    rrf_score = 0.0

    if doc_id in dense_rank_map:
        rrf_score += 1 / (k + dense_rank_map[doc_id])

    if doc_id in bm25_rank_map:
        rrf_score += 1 / (k + bm25_rank_map[doc_id])

    rrf_scores[doc_id] = rrf_score

In [47]:
#sorting the combined results
final_ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

for rank, (idx, score_rrf) in enumerate(final_ranked, start=1):
    print("=" * 80)
    print(f"Final Rank {rank} | Chunk index: {idx} | RRF Score: {score_rrf:.6f}")
    print(chunks[idx][:700])
    print()

Final Rank 1 | Chunk index: 1 | RRF Score: 0.032787
Usage
● FAQ bots
● Knowledge base assistants
● Documentation search
● Customer support
Best when:
● Knowledge is static or slow-changing
● No multi-step reasoning required
2. Agentic RAG
RAG + autonomous decision-making. The system behaves like an agent that
can decide:
● what to retrieve
● whether to retrieve
● which tools to call
● how many steps to take
It’s iterative and dynamic rather than being a single pass.
Example

Final Rank 2 | Chunk index: 11 | RRF Score: 0.031498
--- Page 6 ---
● Long-document QA
● Summarization systems
● Legal / research analysis
Best when:
● Retrieved context is large or noisy
● Fine-grained relevance matters
8. HybridAI RAG
Combines neural methods (embeddings/LLMs) with symbolic or
rule-based systems (knowledge graphs, logic rules).
It enables:
● structured reasoning
● better interpretability
● improved factual consistency
Example
User: “Who is the CEO of the company that owns Instagram?”
System:

Fina

Retrieval + LLM answering
At this stage we will:

take a user question

retrieve top fused chunks

build a prompt with those chunks

send that prompt to an LLM

get a grounded answer

That is where retrieval becomes actual answer generation.


In [48]:
#now choosing the answer model via free groq api
%pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.4 MB/s eta 0:00:00


In [49]:
from groq import Groq
from getpass import getpass

GROQ_API_KEY = getpass("Enter your Groq API key: ")

client = Groq(api_key=GROQ_API_KEY)
MODEL = "llama-3.1-8b-instant"

Enter your Groq API key: ··········


In [50]:
#testing the llm
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Explain RAG in 3 simple lines."}
    ]
)

print(response.choices[0].message.content)

RAG is a color-coded system used to represent the status of projects, tasks, or requirements. 

- R: Red indicates risk or issues that need immediate attention.
- A: Amber represents tasks that need action or follow-up within a specified timeframe.
- G: Green signifies tasks that are complete, on track, or have no issues.


In [51]:
def retrieve_hybrid(query, chunks, model, chunk_embeddings, bm25, top_k=3, k_rrf=60):
    import numpy as np
    from sklearn.metrics.pairwise import cosine_similarity

    query_embedding = model.encode([query], convert_to_numpy=True)
    dense_scores = cosine_similarity(query_embedding, chunk_embeddings)[0]
    dense_indices = np.argsort(dense_scores)[::-1][:top_k]

    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)
    bm25_indices = np.argsort(bm25_scores)[::-1][:top_k]

    dense_rank_map = {doc_id: rank + 1 for rank, doc_id in enumerate(dense_indices)}
    bm25_rank_map = {doc_id: rank + 1 for rank, doc_id in enumerate(bm25_indices)}

    all_candidates = set(dense_indices).union(set(bm25_indices))
    rrf_scores = {}

    for doc_id in all_candidates:
        score = 0.0
        if doc_id in dense_rank_map:
            score += 1 / (k_rrf + dense_rank_map[doc_id])
        if doc_id in bm25_rank_map:
            score += 1 / (k_rrf + bm25_rank_map[doc_id])
        rrf_scores[doc_id] = score

    final_ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    top_chunks = [chunks[idx] for idx, _ in final_ranked[:top_k]]

    return top_chunks, final_ranked[:top_k]

In [52]:
def retrieve_hybrid(query, chunks, model, chunk_embeddings, bm25, top_k=3, k_rrf=60):
    import numpy as np
    from sklearn.metrics.pairwise import cosine_similarity

    query_embedding = model.encode([query], convert_to_numpy=True)
    dense_scores = cosine_similarity(query_embedding, chunk_embeddings)[0]
    dense_indices = np.argsort(dense_scores)[::-1][:top_k]

    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)
    bm25_indices = np.argsort(bm25_scores)[::-1][:top_k]

    dense_rank_map = {doc_id: rank + 1 for rank, doc_id in enumerate(dense_indices)}
    bm25_rank_map = {doc_id: rank + 1 for rank, doc_id in enumerate(bm25_indices)}

    all_candidates = set(dense_indices).union(set(bm25_indices))
    rrf_scores = {}

    for doc_id in all_candidates:
        score = 0.0
        if doc_id in dense_rank_map:
            score += 1 / (k_rrf + dense_rank_map[doc_id])
        if doc_id in bm25_rank_map:
            score += 1 / (k_rrf + bm25_rank_map[doc_id])
        rrf_scores[doc_id] = score

    final_ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

    top_chunks = [
        {
            "chunk_id": int(idx),
            "text": chunks[idx],
            "rrf_score": float(score)
        }
        for idx, score in final_ranked[:top_k]
    ]

    return top_chunks, final_ranked[:top_k]

In [53]:
def answer_with_rag(query, client, llm_model, retriever_model, chunk_embeddings, bm25, chunks, top_k=3):
    top_chunks, ranked = retrieve_hybrid(
        query=query,
        chunks=chunks,
        model=retriever_model,
        chunk_embeddings=chunk_embeddings,
        bm25=bm25,
        top_k=top_k
    )

    context = "\n\n".join(
        [
            f"Context {i+1} | Chunk ID: {chunk['chunk_id']} | RRF Score: {chunk['rrf_score']:.6f}\n{chunk['text']}"
            for i, chunk in enumerate(top_chunks)
        ]
    )

    prompt = f"""
You are a helpful assistant answering questions only from the provided context.

Rules:
- Answer using only the context below.
- If the answer is not clearly in the context, say: "The answer is not available in the provided document."
- Do not use outside knowledge.
- Keep the answer clear and accurate.
- If multiple contexts are retrieved, prefer the most directly relevant one.

Context:
{context}

Question:
{query}

Answer:
"""

    response = client.chat.completions.create(
        model=llm_model,
        messages=[
            {"role": "system", "content": "You answer only from retrieved document context."},
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content, top_chunks, ranked

In [54]:
query = "What is Agentic RAG?"
answer, used_chunks, ranked = answer_with_rag(
    query=query,
    client=client,
    llm_model=MODEL,
    retriever_model=model,
    chunk_embeddings=chunk_embeddings,
    bm25=bm25,
    chunks=chunks,
    top_k=3
)

print("Question:", query)
print("\nAnswer:\n", answer)
print("\nRetrieved chunk ranks:\n", ranked)

for i, chunk in enumerate(used_chunks, start=1):
    print("\n" + "=" * 80)
    print(f"Used Chunk {i}")
    print("Chunk ID:", chunk["chunk_id"])
    print("RRF Score:", chunk["rrf_score"])
    print(chunk["text"][:1000])

Question: What is Agentic RAG?

Answer:
 Agentic RAG is RAG + autonomous decision-making. The system behaves like an agent that can decide: 
- what to retrieve
- whether to retrieve
- which tools to call
- how many steps to take
It’s iterative and dynamic rather than being a single pass.

Retrieved chunk ranks:
 [(np.int64(1), 0.03278688524590164), (np.int64(0), 0.016129032258064516), (np.int64(12), 0.016129032258064516)]

Used Chunk 1
Chunk ID: 1
RRF Score: 0.03278688524590164
Usage
● FAQ bots
● Knowledge base assistants
● Documentation search
● Customer support
Best when:
● Knowledge is static or slow-changing
● No multi-step reasoning required
2. Agentic RAG
RAG + autonomous decision-making. The system behaves like an agent that
can decide:
● what to retrieve
● whether to retrieve
● which tools to call
● how many steps to take
It’s iterative and dynamic rather than being a single pass.
Example

Used Chunk 2
Chunk ID: 0
RRF Score: 0.016129032258064516
--- Page 1 ---
10 RAG Architectu

Memory Building

In [55]:
#simple pythton list memory
chat_history = []

def format_chat_history(chat_history, max_turns=3):
    if not chat_history:
        return ""
    recent = chat_history[-max_turns * 2:]
    formatted = []
    for turn in recent:
        role = "User" if turn["role"] == "user" else "Assistant"
        formatted.append(f"{role}: {turn['content']}")
    return "\n".join(formatted)

In [56]:
#memory aware rag-function
def answer_with_rag_and_memory(query, client, llm_model, retriever_model,
                                chunk_embeddings, bm25, chunks,
                                chat_history, memory_store, top_k=3):

    # ── Step 1: Hybrid document retrieval ──────────────────────────────
    top_chunks, ranked = retrieve_hybrid(
        query=query,
        chunks=chunks,
        model=retriever_model,
        chunk_embeddings=chunk_embeddings,
        bm25=bm25,
        top_k=top_k
    )

    document_context = "\n\n".join([
        f"Doc Context {i+1} | Chunk ID: {chunk['chunk_id']} | RRF Score: {chunk['rrf_score']:.6f}\n{chunk['text']}"
        for i, chunk in enumerate(top_chunks)
    ])

    # ── Step 2: FAISS memory search ────────────────────────────────────
    memory_results = memory_store.search_memory(query, top_k=2)

    if memory_results:
        memory_context = "\n\n".join([
            f"Past Memory {i+1} (Distance: {r['distance']:.4f}):\n{r['memory_text']}"
            for i, r in enumerate(memory_results)
        ])
    else:
        memory_context = ""

    # ── Step 3: Recent chat history ────────────────────────────────────
    recent_history = format_chat_history(chat_history, max_turns=3)

    # ── Step 4: Build prompt with all three inputs ─────────────────────
    memory_section = f"""
Relevant Past Conversation (from memory search):
{memory_context}
""" if memory_context else ""

    history_section = f"""
Recent Conversation History:
{recent_history}
""" if recent_history else ""

    prompt = f"""
You are a helpful assistant answering questions only from the provided document context.

Rules:
- Answer using only the document context below.
- If the answer is not in the document context, say: "The answer is not available in the provided document."
- Do not use outside knowledge.
- Use past memory and conversation history only to understand follow-up questions.
- Never answer from memory alone. Always ground in document context.
{memory_section}{history_section}
Document Context:
{document_context}

Question:
{query}

Answer:
"""

    # ── Step 5: Call LLM ───────────────────────────────────────────────
    response = client.chat.completions.create(
        model=llm_model,
        messages=[
            {"role": "system", "content": "You answer only from retrieved document context. Use memory and history only to understand the question better."},
            {"role": "user", "content": prompt}
        ]
    )

    answer = response.choices[0].message.content

    # ── Step 6: Update both memory systems ─────────────────────────────
    chat_history.append({"role": "user", "content": query})
    chat_history.append({"role": "assistant", "content": answer})
    memory_store.add_memory(query, answer)

    return answer, top_chunks, ranked, chat_history

In [57]:
chat_history = []
memory_store.clear()

questions = [
    "What is Agentic RAG?",
    "How is it different from standard RAG?",
    "Give me an example of when to use it.",
    "Which one is better for enterprise search?"
]

for q in questions:
    print("=" * 80)
    print("User:", q)
    answer, used_chunks, ranked, chat_history = answer_with_rag_and_memory(
        query=q,
        client=client,
        llm_model=MODEL,
        retriever_model=model,
        chunk_embeddings=chunk_embeddings,
        bm25=bm25,
        chunks=chunks,
        chat_history=chat_history,
        memory_store=memory_store,
        top_k=3
    )
    print("Assistant:", answer)
    print(f"[Memory store size: {len(memory_store.memories)}]")
    print()

User: What is Agentic RAG?
Assistant: Agentic RAG is described as RAG + autonomous decision-making. The system behaves like an agent that can decide:

● what to retrieve
● whether to retrieve
● which tools to call
● how many steps to take

It’s iterative and dynamic rather than being a single pass.
[Memory store size: 1]

User: How is it different from standard RAG?
Assistant: Based on the document context, I will answer your follow-up question. 

According to the document context (Chunk ID: 1), Agentic RAG is different from Standard RAG in that it has autonomous decision-making capabilities and behaves like an agent. This allows it to be iterative and dynamic, making multiple steps rather than operating as a single pass.

Here's an excerpt from the document context:
"2. Agentic RAG
RAG + autonomous decision-making. The system behaves like an agent that
can decide:
● what to retrieve
● whether to retrieve
● which tools to call
● how many steps to take
It’s iterative and dynamic rather 

In [61]:
# test with follow up questions
chat_history = []

questions = [
    "What is Agentic RAG?",
    "How is it different from standard RAG?",
    "Give me an example of when to use it."
]

for q in questions:
    print("=" * 80)
    print("User:", q)
    answer, used_chunks, ranked, chat_history = answer_with_rag_and_memory(
        query=q,
        client=client,
        llm_model=MODEL,
        retriever_model=model,
        chunk_embeddings=chunk_embeddings,
        bm25=bm25,
        chunks=chunks,
        chat_history=chat_history,
        memory_store=memory_store,
        top_k=3
    )
    print("Assistant:", answer)
    print(f"[Memory store size: {len(memory_store.memories)}]")
    print()

User: What is Agentic RAG?
Assistant: Agentic RAG is described as RAG + autonomous decision-making. The system behaves like an agent that can decide:

● what to retrieve
● whether to retrieve
● which tools to call
● how many steps to take

It’s iterative and dynamic rather than being a single pass.
[Memory store size: 5]

User: How is it different from standard RAG?
Assistant: Based on the document context (Chunk ID: 1), Agentic RAG is different from Standard RAG in that it has autonomous decision-making capabilities and behaves like an agent. This allows it to be iterative and dynamic, making multiple steps rather than operating as a single pass.

Here's an excerpt from the document context:
"2. Agentic RAG
RAG + autonomous decision-making. The system behaves like an agent that
can decide:
● what to retrieve
● whether to retrieve
● which tools to call
● how many steps to take
It’s iterative and dynamic rather than being a single pass."
[Memory store size: 6]

User: Give me an example 

In [60]:
#Adding Faiss also.
%pip install -q faiss-cpu

In [62]:
import os
import json
import faiss
import numpy as np

In [63]:
#memory manager class
class FaissMemoryStore:
    def __init__(self, embed_model, dim=None,
                 index_path="memory_index.faiss",
                 store_path="memory_store.json"):
        self.embed_model = embed_model
        self.index_path = index_path
        self.store_path = store_path

        if dim is None:
            test_vec = self.embed_model.encode(["test"], convert_to_numpy=True)
            dim = test_vec.shape[1]

        self.dim = dim
        self.memories = []

        if os.path.exists(self.index_path) and os.path.exists(self.store_path):
            self.index = faiss.read_index(self.index_path)
            with open(self.store_path, "r", encoding="utf-8") as f:
                self.memories = json.load(f)
        else:
            self.index = faiss.IndexFlatL2(self.dim)
            self.memories = []

    def save(self):
        faiss.write_index(self.index, self.index_path)
        with open(self.store_path, "w", encoding="utf-8") as f:
            json.dump(self.memories, f, ensure_ascii=False, indent=2)

    def add_memory(self, user_query, assistant_answer):
        memory_text = f"User: {user_query}\nAssistant: {assistant_answer}"

        embedding = self.embed_model.encode([memory_text], convert_to_numpy=True)
        embedding = np.array(embedding).astype("float32")

        self.index.add(embedding)

        memory_item = {
            "id": len(self.memories),
            "user": user_query,
            "assistant": assistant_answer,
            "memory_text": memory_text
        }

        self.memories.append(memory_item)
        self.save()

    def search_memory(self, query, top_k=3):
        if len(self.memories) == 0 or self.index.ntotal == 0:
            return []

        query_embedding = self.embed_model.encode([query], convert_to_numpy=True)
        query_embedding = np.array(query_embedding).astype("float32")

        distances, indices = self.index.search(query_embedding, top_k)

        results = []
        for idx, dist in zip(indices[0], distances[0]):
            if idx != -1 and idx < len(self.memories):
                item = self.memories[idx].copy()
                item["distance"] = float(dist)
                results.append(item)

        return results

    def clear(self):
        self.index = faiss.IndexFlatL2(self.dim)
        self.memories = []
        self.save()

In [64]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [65]:
#initializing the memory
memory_store = FaissMemoryStore(embed_model=model)

In [66]:
#verification cell
print("Memory dimension:", memory_store.dim)
print("Current memory count:", len(memory_store.memories))
print("FAISS total vectors:", memory_store.index.ntotal)

Memory dimension: 384
Current memory count: 7
FAISS total vectors: 7


In [67]:
memory_store.add_memory(
    "What is Agentic RAG?",
    "Agentic RAG is RAG plus autonomous decision-making."
)

memory_store.add_memory(
    "How is it different from standard RAG?",
    "Standard RAG is a baseline retrieval setup, while Agentic RAG can decide what to retrieve and how many steps to take."
)

memory_store.add_memory(
    "Give me an example of when to use it.",
    "Use Agentic RAG when retrieval quality is unreliable or when the system must decide multiple retrieval steps."
)

Test Retrieval

In [68]:
results = memory_store.search_memory(
    "Explain difference between Agentic RAG and standard RAG",
    top_k=2
)

for r in results:
    print("=" * 80)
    print("ID:", r["id"])
    print("Distance:", r["distance"])
    print(r["memory_text"])

ID: 8
Distance: 0.39713937044143677
User: How is it different from standard RAG?
Assistant: Standard RAG is a baseline retrieval setup, while Agentic RAG can decide what to retrieve and how many steps to take.
ID: 5
Distance: 0.5580523014068604
User: How is it different from standard RAG?
Assistant: Based on the document context (Chunk ID: 1), Agentic RAG is different from Standard RAG in that it has autonomous decision-making capabilities and behaves like an agent. This allows it to be iterative and dynamic, making multiple steps rather than operating as a single pass.

Here's an excerpt from the document context:
"2. Agentic RAG
RAG + autonomous decision-making. The system behaves like an agent that
can decide:
● what to retrieve
● whether to retrieve
● which tools to call
● how many steps to take
It’s iterative and dynamic rather than being a single pass."


In [69]:
#verifying memory methods.
# test that both methods exist and work
print(type(memory_store))
print(dir(memory_store))

<class '__main__.FaissMemoryStore'>
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'add_memory', 'clear', 'dim', 'embed_model', 'index', 'index_path', 'memories', 'save', 'search_memory', 'store_path']


In [70]:
# check what add_memory expects as input
import inspect
print(inspect.signature(memory_store.add_memory))
print(inspect.signature(memory_store.search_memory))

(user_query, assistant_answer)
(query, top_k=3)
